# Acoustic Insect Species Identification — Feature Pipeline & EDA
### ICT-4442 Deep Learning Mini Project | Author: Saurabh Tiwari (230911238)

This notebook demonstrates:
1. Insect bioacoustic signal synthesis & structure (Orthoptera & Cicadidae pulse trains)
2. 128-band **Log-Mel Spectrogram Extraction**
3. **Handcrafted Acoustic Feature Extraction** (MFCCs, Spectral Centroid, Bandwidth, Rolloff, ZCR)
4. **Data Augmentations**: Waveform perturbations & **SpecAugment** (Frequency & Time Masking)
5. Temperature dependence according to **Dolbear's Law**

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

from src.data.features import LogMelExtractor, HandcraftedFeatureExtractor
from src.data.augmentation import WaveformAugmentation, SpecAugment
from src.data.dataset import SyntheticInsectDataset

## 1. Bioacoustic Signal Structure & Synthetic Insect Calls
Insects produce sound via **stridulation** (rubbing specialized body parts together) resulting in periodic pulse trains.

In [ ]:
dataset = SyntheticInsectDataset(num_samples=20, num_classes=5, chunk_duration=5.0, random_seed=42)
sample = dataset[0]

spec = sample['spectrogram'] # (1, 128, time_steps)
label = sample['label'].item()
temp = sample['raw_temperature'].item()

print(f"Sample Species Label: {label}, Ambient Temperature: {temp:.1f} °C")
print(f"Spectrogram Shape: {spec.shape} -> (Channels, Mel-Bands, Time-Frames)")

## 2. Visualizing Log-Mel Spectrogram (128 Bands)

In [ ]:
plt.figure(figsize=(10, 4))
plt.imshow(spec.squeeze().numpy(), origin='lower', aspect='auto', cmap='viridis')
plt.colorbar(format='%+2.0f dB')
plt.title(f"128-Band Log-Mel Spectrogram (Species {label}, Temp {temp:.1f}°C)", fontsize=13, fontweight='bold')
plt.xlabel("Time Frame (Hop size = 512)", fontsize=11)
plt.ylabel("Mel Frequency Bin (0 - 128)", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Data Augmentation: SpecAugment in Action
Applies random frequency and time masking to prevent overfitting to citizen-science microphone artifacts.

In [ ]:
spec_aug = SpecAugment(freq_mask_param=20, time_mask_param=35, num_freq_masks=2, num_time_masks=2)
spec_aug.train(True)
augmented_spec = spec_aug(spec.unsqueeze(0)).squeeze(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(spec.squeeze().numpy(), origin='lower', aspect='auto', cmap='magma')
axes[0].set_title("Original Log-Mel Spectrogram", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Time Steps")
axes[0].set_ylabel("Mel Bins")

axes[1].imshow(augmented_spec.squeeze().numpy(), origin='lower', aspect='auto', cmap='magma')
axes[1].set_title("SpecAugment Masked Spectrogram (2 Freq + 2 Time Masks)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Time Steps")
axes[1].set_ylabel("Mel Bins")

plt.tight_layout()
plt.show()

## 4. Handcrafted Feature Extraction for Classical Baseline
Extracts 49 bioacoustic statistical features: MFCCs, Deltas, Spectral Centroid, Bandwidth, Rolloff, and ZCR.

In [ ]:
raw_audio = dataset._synthesize_insect_call(species_id=label, temperature=temp)
hc_extractor = HandcraftedFeatureExtractor(sample_rate=22050)
features = hc_extractor.extract(raw_audio)

print(f"Extracted {len(features)} Handcrafted Features:")
print("MFCC Means (1-13):", np.round(features[:13], 2))
print("Spectral Centroid (mean, std):", np.round(features[39:41], 2))
print("Spectral Bandwidth (mean, std):", np.round(features[41:43], 2))
print("Zero-Crossing Rate (mean, std):", np.round(features[45:47], 4))